In [ ]:
# @title
import pandas as pd
import numpy as np


pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)


def saisie(texte):
    while True:
        try:
            x = int(input(texte))
            if 0 <= x <= 5:
                return x

            print("Un nombre entre 0 et 5 !")
        except ValueError :
            print("Ce n'est pas un nombre valide, essayez encore.")


def log_per_capita(mon_df, ma_colonne, colonne_population):

    mon_df[ma_colonne] = np.log1p(mon_df[ma_colonne]/mon_df[colonne_population])

    return mon_df


def fonction_z_score(mon_df, ma_colonne):

    nom_générique_variable = f"{ma_colonne}_transitoire"

    mon_df[nom_générique_variable] = (mon_df[ma_colonne] - mon_df[ma_colonne].mean()) / mon_df[ma_colonne].std()

    return mon_df


def changement_polarité(mon_df, ma_colonne):

    nom_générique_variable = f"{ma_colonne}_transitoire"

    mon_df[nom_générique_variable] = mon_df[nom_générique_variable] * (-1)

    return mon_df


def fonction_norm(mon_df, ma_colonne):

    nom_générique_variable = f"{ma_colonne}_transitoire"

    mon_df[nom_générique_variable] = (mon_df[nom_générique_variable] - mon_df[nom_générique_variable].min()) / (mon_df[nom_générique_variable].max() - mon_df[nom_générique_variable].min())

    return mon_df


def fonction_mulitplication_notes(mon_df, ma_colonne, note):

    nom_générique_variable = f"{ma_colonne}_transitoire"

    mon_df[nom_générique_variable] = mon_df[nom_générique_variable] * note

    return mon_df

# Les jeux de données

df_prix_immobilier_raw = pd.read_csv("/boot/prix_immobilier.csv", sep = ";")
df_tp_raw = pd.read_csv("/boot/tp.csv", sep = ";")
df_commerces_raw = pd.read_csv("/boot/commerces.csv", sep = ";")
df_qualité_air_raw = pd.read_csv("/boot/qualité_air.csv", sep = ";")
df_places_sport_raw = pd.read_csv("/boot/places_sport.csv", sep = ";")
df_fisc_raw = pd.read_csv("/boot/fisc.csv", sep = ";")
df_logements_vacants_raw = pd.read_csv("/boot/logements_vacants.csv", sep = ";")
df_pop_raw = pd.read_csv("/boot/pop_commune.csv", sep= ";", encoding="cp1252")

df_matrice_communes_ofs_raw = pd.read_csv("/boot/matrice_commun1e_ofs.csv", sep = ";")



while True:
    # Les jeux de données copiés

    df_prix_immobilier = df_prix_immobilier_raw.copy()
    df_tp = df_tp_raw.copy()
    df_commerces = df_commerces_raw.copy()
    df_qualité_air = df_qualité_air_raw.copy()
    df_places_sport = df_places_sport_raw.copy()
    df_fisc = df_fisc_raw.copy()
    df_logements_vacants = df_logements_vacants_raw.copy()
    df_pop = df_pop_raw.copy()

    df_matrice_communes_ofs = df_matrice_communes_ofs_raw.copy()
    #****************************************************************
    # Les notes
    print("                -----> HabitatScore 2026 <------")
    print("Pour les questions suivantes, prière de donner un importance entre 1 et 5")
    print("-------------------------------------------------------------------------")
    print()

    a = saisie("Quelle importance donnez-vous à l'accessibilité pour l'achat d'un appartement ?")
    b = saisie("Quelle importance donnez-vous à l'accessibilité pour l'achat d'une maison ?")
    c = saisie("Quelle importance donnez-vous à l'accessibilité des transports publiques ?")
    d = saisie("Quelle importance donnez-vous à l'accessibilité des commodités (commerces etc.) ?")
    e = saisie("Quelle importance donnez-vous à la qualité de l'air ?")
    f = saisie("Quelle importance donnez-vous à l'accessibilité des places de sport et de divertissements ?")
    g = saisie("Quelle importance donnez-vous aux taux d'imposition communaux ?")
    h = saisie("Quelle importance donnez-vous à l'accessibilité logements de manière globale ?")


    #****************************************************************
    # Transofrmation/Travail df

    # Prix immobilier
    df_prix_immobilier_appartement = df_prix_immobilier.copy()
    df_prix_immobilier_appartement["Moyenne1"] = (df_prix_immobilier["Appart. Prix bas"] + df_prix_immobilier["Appart. Prix haut"])/2
    df_prix_immobilier_maison = df_prix_immobilier.copy()
    df_prix_immobilier_maison["Moyenne2"] = (df_prix_immobilier["Maison prix bas"] + df_prix_immobilier["Maison prix haut"])/2

    # tp

    df_tp["Commune"] = df_tp["Commune"].str.replace(r'\s*\(.*?\)', '', regex=True).str.strip()


    poids = {"METRO" : 1,"TRAIN": 2, "BUS|METRO" : 3, "BUS|TRAM" : 4, "BUS|METRO|TRAM" : 5, "BUS|TRAIN" : 666}
    df_tp1 = df_tp[['Commune', 'Moyen de transport']].copy()
    df_tp1["Poids"] = df_tp1["Moyen de transport"].map(poids).fillna(1)
    df_score_tp = df_tp1.groupby("Commune")["Poids"].sum().reset_index()

    df_score_tp["Commune"] = df_score_tp["Commune"].str.split('(').str[0].str.strip()

    # Traitement des donnes du nb logements vacants, de str vers int

    df_logements_vacants["Total logements vacants"] = df_logements_vacants["Total logements vacants"].fillna("0").str.replace("-","0").astype(int)


    # Traitement de population et fusion


    # 1. Extraction du numéro depuis la colonne
    df_pop['No_ofs'] = df_pop['ofs'].str[6:10]

    # 2. Filtrage pour conserver uniquement le canton de Vaud (enlever?)
    df_pop_vaud = df_pop[(df_pop["No_ofs"] >= "5401") & (df_pop["No_ofs"] <= "5939")].copy()

    # 3. Alignement des types : Conversion des clés en nombres entiers (int) pour éviter les conflits
    df_pop_vaud['No_ofs'] = pd.to_numeric(df_pop_vaud['No_ofs'], errors='coerce').astype(int)
    df_matrice_communes_ofs['A'] = pd.to_numeric(df_matrice_communes_ofs['A'], errors='coerce').astype(int)

    # 4. Fusion finale par jointure interne
    df_pop_vaud = df_pop_vaud.merge(
        df_matrice_communes_ofs[['A', 'Communes']],
        left_on='No_ofs',
        right_on='A',
        how="inner")


    #****************************************************************
    #Merge des df

    #Merge 1
    df_merge = pd.merge(df_prix_immobilier_appartement[["Commune", "Moyenne1"]], df_prix_immobilier_maison[["Commune", "Moyenne2"]], on = "Commune", how = "inner")
    #Merge2
    df_merge = pd.merge(df_merge, df_score_tp[["Poids", "Commune"]], on = "Commune", how = "inner")
    #Merge 3
    df_merge = pd.merge(df_merge, df_commerces, on = "Commune", how = "inner")
    #Merge 4
    df_merge = pd.merge(df_merge, df_qualité_air[["Commune", "Indice (IQA)"]] , on = "Commune", how = "inner")
    #Merge 5
    df_merge = pd.merge(df_merge, df_places_sport[["Commune", "Total places"]] , on = "Commune", how = "inner")
    #Merge 6
    df_merge = pd.merge(df_merge, df_fisc[["Commune","Impôt rentrée argent"]], on = "Commune", how = "inner")
    #Merge 7
    df_merge = pd.merge(df_merge, df_logements_vacants[["Commune", "Total logements vacants"]], on = "Commune", how = "inner")
    #Merge 8
    df_merge = pd.merge(df_merge, df_pop_vaud, left_on="Commune", right_on="Communes", how="inner")

    #****************************************************************
    #log_per_capita


    #Desserte TP
    df_merge = log_per_capita(df_merge, "Poids", "Population")
    #Commerces
    df_merge = log_per_capita(df_merge, "nb établissements commerciaux", "Population")
    #Place de sport/jeux
    df_merge = log_per_capita(df_merge, "Total places", "Population")
    #Logements disponible
    df_merge = log_per_capita(df_merge, "Total logements vacants", "Population")


    #***********************************************************
    #Z_scores

    df_merge = fonction_z_score(df_merge, "Moyenne1")
    df_merge = fonction_z_score(df_merge, "Moyenne2")
    df_merge = fonction_z_score(df_merge, "Poids")
    df_merge = fonction_z_score(df_merge, "nb établissements commerciaux")
    df_merge = fonction_z_score(df_merge, "Indice (IQA)")
    df_merge = fonction_z_score(df_merge, "Total places")
    df_merge = fonction_z_score(df_merge, "Impôt rentrée argent")
    df_merge = fonction_z_score(df_merge, "Total logements vacants")


    #***********************************************************
    #Polarité variables


    # Identification et traitement des dimensions ayant un impact négatif -> *(-1)

    #Accessiblité maisons
    #Accessibilité appartements
    #Coefficient fiscaux communaux

    df_merge = changement_polarité(df_merge, "Moyenne1")
    df_merge = changement_polarité(df_merge, "Moyenne2")
    df_merge = changement_polarité(df_merge, "Impôt rentrée argent")


    #***********************************************************
    #Normalisation

    df_merge = fonction_norm(df_merge, "Moyenne1")
    df_merge = fonction_norm(df_merge, "Moyenne2")
    df_merge = fonction_norm(df_merge, "Poids")
    df_merge = fonction_norm(df_merge, "nb établissements commerciaux")
    df_merge = fonction_norm(df_merge, "Indice (IQA)")
    df_merge = fonction_norm(df_merge, "Total places")
    df_merge = fonction_norm(df_merge, "Impôt rentrée argent")
    df_merge = fonction_norm(df_merge, "Total logements vacants")

    #***********************************************************
    #Multiplication


    df_merge = fonction_mulitplication_notes(df_merge, "Moyenne1", a)
    df_merge = fonction_mulitplication_notes(df_merge, "Moyenne2", b)
    df_merge = fonction_mulitplication_notes(df_merge, "Poids", c)
    df_merge = fonction_mulitplication_notes(df_merge, "nb établissements commerciaux", d)
    df_merge = fonction_mulitplication_notes(df_merge, "Indice (IQA)", e)
    df_merge = fonction_mulitplication_notes(df_merge, "Total places", f)
    df_merge = fonction_mulitplication_notes(df_merge, "Impôt rentrée argent", g)
    df_merge = fonction_mulitplication_notes(df_merge, "Total logements vacants", h)

    df_merge.to_csv("blblba.csv", index = False, encoding = "utf-8-sig")
    #***********************************************************
    #Fin

    liste_en_têtes = ["Moyenne1_transitoire", "Moyenne2_transitoire",
                      "Poids_transitoire", "nb établissements commerciaux_transitoire", "Indice (IQA)_transitoire",
                      "Total places_transitoire", "Impôt rentrée argent_transitoire", "Total logements vacants_transitoire"]

    liste_justification = ["Accessibilité achat appartement",
                           "Accessibilité achat maison",
                           "Offre transports publiques",
                           "Offre commerces et services",
                           "Qualité de l'air",
                           "Offre place de sports et loisirs",
                           "Charge fiscale modérée",
                           "Nomre de logements à louer"]

    dictionnaire_justification = dict(zip(liste_en_têtes, liste_justification))



    df_merge["Score_final"] = df_merge[liste_en_têtes].sum(axis = 1)
    df_merge["Justification"] = df_merge[liste_en_têtes ].idxmax(axis = 1)
    df_merge["Justification"] = df_merge["Justification"].map(dictionnaire_justification)

    top_max = df_merge.nlargest(10, "Score_final")[["Commune", "Justification"]].reset_index()
    top_max["index"] = top_max.index + 1
    top_max = top_max.rename(columns = {"index" : "Classement de la commune"})

    print()
    print("                 Les meilleures communes HabitatScore sélectionnées pour vous")
    print()
    print("                                 Et vos communes sont ..... !")

    Hauteur = 20
    décalage = 21
    k = 0
    while k <= Hauteur+1:

        d = 0
        while d < décalage :
            print(" ", end = "")
            d+=1


        i=0
        while i <= k :
            print(" ", end = "")
            i+=1


        i = 2 * Hauteur -1
        while i >= 2*k-1:
            print("*", end="")
            i-=1

        i=0
        while i <= k :
            print(" ", end = "")
            i+=1


        print()
        k+=1


    print(top_max.to_string(index=False))

    print()
    refaire= input ("Voulez vous réessayer une simulation ? O/N :")

    if refaire.lower() != 'o':

        print("Merci d'avoir utiliser cette simulation !")
        print("HabitatScore Mai 2026")
        break




















